# A/B Test Design & Simulation – Reviews on the Product Page
**Project:** E-commerce Funnel and A/B Testing Analysis (GA4)
**Data:** Google Merchandise Store GA4 public sample (BigQuery), 26 Nov 2020 – 31 Jan 2021

**Business problem:** The funnel analysis (see `sql/funnel_analysis.sql`) showed the product page
is the biggest drop-off: only 26.7% of product viewers add to cart. First-time visitors
(~90% of product viewers) add to cart at 25.2% vs 39.2% for returning visitors.

**Hypothesis:** If we show ratings and reviews near the "Add to cart" button, more product
viewers will add to cart, because first-time visitors need trust in a store they don't know.

**What this notebook does:**
1. Gets the real baseline from BigQuery
2. Calculates the sample size the test needs
3. Simulates one A/B test and analyses it
4. Repeats the test 1,000 times to check the design works

⚠️ *The experiment is **simulated**. The baseline comes from real GA4 data; the +10% effect is an assumption.*

## Step 1 – Connect to BigQuery

In [9]:
from google.colab import auth
auth.authenticate_user()

## Step 2 – Get the baseline from BigQuery
**What:** Count product viewers, and how many of them added to cart and purchased (26 Nov 2020 – 31 Jan 2021).

**Why:** The A/B test needs a starting rate (the baseline). Taking it directly from BigQuery means no number is typed by hand, and anyone can trace where it comes from.

**Population:** only users who viewed a product, because that's who the test would include.

In [10]:
from google.cloud import bigquery

client = bigquery.Client(project="funnel-portfolio-514915")

sql = """
WITH user_steps AS (
  SELECT
    user_pseudo_id,
    MAX(IF(event_name = 'view_item',   1, 0)) AS viewed_item,
    MAX(IF(event_name = 'add_to_cart', 1, 0)) AS added_to_cart,
    MAX(IF(event_name = 'purchase',    1, 0)) AS purchased
  FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '20201126' AND '20210131'
  GROUP BY user_pseudo_id
)
SELECT
  SUM(viewed_item)                   AS viewers,
  SUM(viewed_item * added_to_cart)   AS viewers_added_to_cart,
  SUM(viewed_item * purchased)       AS viewers_purchased
FROM user_steps
"""

df = client.query(sql).to_dataframe()
df

,viewers,viewers_added_to_cart,viewers_purchased
0,45107,12017,3328


**Findings:** 45,107 users viewed a product. 12,017 of them added to cart (baseline 26.64%)
and 3,328 purchased (7.38%, the guardrail baseline).
These are slightly lower than the full funnel (12,023 add-to-cart) because a few users
added to cart without a product-view event; I count only product viewers, the test population.

## Step 3 – How many users does the test need? (sample size)
**What:** Use the baseline from Step 2 to calculate how many users each group needs, and roughly how many days the test would run.

**Why:** A test that is too small can miss a real improvement or mistake luck for an effect. The size is decided before the test starts, so the result can be trusted.

**My decisions (inputs):**
- **Baseline:** 26.64% of product viewers add to cart (from Step 2)
- **Minimum detectable effect (MDE):** +10% relative → target 29.31%. The smallest gain worth detecting.
- **Significance level (α):** 5% – the accepted risk of a false alarm (saying "it works" when it doesn't).
- **Power:** 80% – the chance of detecting the +10% effect if it is real.

In [11]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
import math

# 1. Baseline taken from the BigQuery result (df), not typed by hand
viewers       = int(df.loc[0, "viewers"])
added_to_cart = int(df.loc[0, "viewers_added_to_cart"])
purchased     = int(df.loc[0, "viewers_purchased"])

baseline           = added_to_cart / viewers   # primary metric: view → cart
guardrail_baseline = purchased / viewers       # guardrail: purchase conversion

# 2. Test settings (our decisions)
mde_relative = 0.10
target = baseline * (1 + mde_relative)
alpha  = 0.05
power  = 0.80

# 3. Sample size per group
effect = proportion_effectsize(target, baseline)
n_per_group = math.ceil(NormalIndPower().solve_power(effect_size=effect,
                                                     alpha=alpha, power=power, ratio=1))

# 4. Rough test duration (67 days = 26 Nov 2020 – 31 Jan 2021)
daily_viewers = viewers / 67
days = (2 * n_per_group) / daily_viewers

print(f"Baseline view→cart:  {baseline:.2%}")
print(f"Target view→cart:    {target:.2%}")
print(f"Guardrail baseline:  {guardrail_baseline:.2%}")
print(f"Users per group:     {n_per_group:,}")
print(f"Total users:         {2 * n_per_group:,}")
print(f"Estimated days:      {days:.1f}")
print(f"Product viewers per day: {daily_viewers:,.0f}")

Baseline view→cart:  26.64%
Target view→cart:    29.31%
Guardrail baseline:  7.38%
Users per group:     4,454
Total users:         8,908
Estimated days:      13.2
Product viewers per day: 673


**Findings:** To detect a +10% lift (26.64% → 29.31%) with 5% significance and 80% power,
the test needs **4,454 product viewers per group (8,908 in total)**.
With about **673 product viewers per day** in this period, that is **~13 days → plan 14 days**
(two full weeks, so both weekdays and weekends are included).
The guardrail, purchase conversion, starts at **7.38%**.

## Step 4 – Simulate one A/B test
**What:** Create 8,908 simulated product viewers, split them randomly into control
(current page) and treatment (page with reviews), and simulate who adds to cart and purchases.

**Why:** We have no real experiment data, so we simulate one to practise the full analysis.

**Assumptions (set by me, not from data):**
- Control adds to cart at the real baseline (26.64%).
- Treatment adds to cart at baseline +10% (29.31%) – the effect we're testing for.
- Purchase rate is the same in both groups (7.38%) and independent of add-to-cart (simplification).
- Random seed 42, so anyone who runs the notebook gets the same result.

In [12]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)       # fixed seed = same "random" result every run

true_lift   = 0.10                    # ASSUMPTION: reviews raise add-to-cart by 10%
total_users = 2 * n_per_group

# 1. Randomly assign each user to control or treatment (like a coin flip)
group = rng.choice(["control", "treatment"], size=total_users)

# 2. Each user's chance of adding to cart depends on their group
p_cart = np.where(group == "treatment", baseline * (1 + true_lift), baseline)
added  = rng.random(total_users) < p_cart

# 3. Purchase (guardrail): same chance in both groups (assumption)
purchased = rng.random(total_users) < guardrail_baseline

# 4. Put it in a table: one row per simulated user
sim = pd.DataFrame({
    "user_id":       np.arange(total_users),
    "group":         group,
    "added_to_cart": added.astype(int),
    "purchased":     purchased.astype(int),
})

# 5. Summary per group
summary = sim.groupby("group").agg(
    users=("user_id", "count"),
    added_to_cart=("added_to_cart", "sum"),
    purchased=("purchased", "sum"),
)
summary["cart_rate_%"]     = (100 * summary["added_to_cart"] / summary["users"]).round(2)
summary["purchase_rate_%"] = (100 * summary["purchased"] / summary["users"]).round(2)

print(sim.head())
summary

   user_id      group  added_to_cart  purchased
0        0    control              0          0
1        1  treatment              0          0
2        2  treatment              1          0
3        3    control              0          0
4        4    control              1          0


,users,added_to_cart,purchased,cart_rate_%,purchase_rate_%
group,,,,,
control,4505,1244,322,27.61,7.15
treatment,4403,1283,325,29.14,7.38


**Findings:** Control: 4,505 users, 27.61% added to cart. Treatment: 4,403 users, 29.14%.

Neither group matches its set rate exactly (26.64% and 29.31%), and the groups are not exactly 50/50. That is normal random variation, like flipping a coin 100 times and not getting exactly 50 heads. It is why the next step uses statistical tests instead of comparing the two rates by eye.

## Step 5 – Analyse the simulated test
**What:** Run the same checks an analyst would run on a real A/B test:
1. **SRM check (sample ratio mismatch):** is the group split consistent with a random 50/50? If not, the randomisation is broken and no other result can be trusted.

2. **Two-proportion z-test** on add-to-cart: is the difference between groups real, or could it be luck?

3. **95% confidence interval:** the range where the true difference probably lies.

4. **Guardrail:** did purchase conversion get worse?

**Why:** Comparing two rates by eye is not enough, because random variation alone creates gaps.

**Decision rule (set BEFORE looking at the results):** ship only if
SRM is OK, p < 0.05, the confidence interval is fully above 0, and the guardrail is not worse.

In [13]:
from scipy.stats import chisquare
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

c = summary.loc["control"]
t = summary.loc["treatment"]

# 1. SRM check: compare actual split with an expected 50/50 split
srm_p = chisquare([c["users"], t["users"]]).pvalue
print(f"SRM p-value: {srm_p:.3f}  ->  {'OK' if srm_p >= 0.01 else 'PROBLEM: check randomisation'}")

# 2. Primary metric: add to cart
z, p_value = proportions_ztest([t["added_to_cart"], c["added_to_cart"]],
                               [t["users"],         c["users"]])
low, high = confint_proportions_2indep(t["added_to_cart"], t["users"],
                                       c["added_to_cart"], c["users"],
                                       compare="diff", method="wald")
diff = t["cart_rate_%"] - c["cart_rate_%"]
print(f"View→cart: control {c['cart_rate_%']}%, treatment {t['cart_rate_%']}%, difference {diff:.2f} pts")
print(f"p-value: {p_value:.3f}   95% CI of difference: {100*low:.2f} to {100*high:.2f} pts")

# 3. Guardrail: purchase conversion
_, p_guard = proportions_ztest([t["purchased"], c["purchased"]], [t["users"], c["users"]])
guard_ok = not (t["purchase_rate_%"] < c["purchase_rate_%"] and p_guard < 0.05)
print(f"Guardrail purchase: control {c['purchase_rate_%']}%, treatment {t['purchase_rate_%']}%, p = {p_guard:.3f}  ->  {'OK' if guard_ok else 'WORSE'}")

# 4. Decision (rule fixed before looking at the result)
ship = (srm_p >= 0.01) and (p_value < 0.05) and (low > 0) and guard_ok
print("\nDECISION:", "SHIP the reviews" if ship else "DO NOT SHIP (no clear evidence of improvement)")

SRM p-value: 0.280  ->  OK
View→cart: control 27.61%, treatment 29.14%, difference 1.53 pts
p-value: 0.110   95% CI of difference: -0.35 to 3.40 pts
Guardrail purchase: control 7.15%, treatment 7.38%, p = 0.671  ->  OK

DECISION: DO NOT SHIP (no clear evidence of improvement)


**Findings:**
- **SRM:** p = 0.280 → the 4,505 / 4,403 split is normal for random assignment. Results can be trusted.
- **Add-to-cart:** treatment was **+1.53 points** higher (29.14% vs 27.61%), but **p = 0.110**,
  above 0.05. A gap this size could happen by chance.
- **95% CI:** **−0.35 to +3.40 points**. It includes 0, so "no effect" cannot be ruled out.
- **Guardrail:** purchase conversion 7.15% vs 7.38% (p = 0.671) → not worse.

**Decision: DO NOT SHIP – inconclusive.** "No clear evidence of improvement", not "reviews don't work".

**Interesting point:** a real +10% effect was built into this simulation, and this single test
still missed it, because control came out high by chance (27.61% instead of 26.64%).
This is the ~20% risk accepted with 80% power. Step 6 checks how often that happens.

## Step 6 – Repeat the experiment 1,000 times
**What:** Run the full simulated test (random split → add-to-cart → z-test) 1,000 times,
and count how often the test says "significant" (p < 0.05). Done twice:
- **With a real +10% effect** → % significant = **power** (planned: 80%)
- **With no effect (0%)** → % significant = **false-alarm rate** (planned: 5%)

**Why:** One test can be lucky or unlucky (Step 5 was unlucky). Repeating it many times
shows how reliable the test design is, and checks that the sample size from Step 3 really
gives the power and false-alarm rate I planned.

In [14]:
def run_many(true_lift, n_runs=1000, seed=0):
    rng = np.random.default_rng(seed)
    significant = 0                                   # counter

    for _ in range(n_runs):                           # repeat n_runs times
        n_c = rng.binomial(2 * n_per_group, 0.5)      # random group split
        n_t = 2 * n_per_group - n_c
        cart_c = rng.binomial(n_c, baseline)                     # control adds to cart
        cart_t = rng.binomial(n_t, baseline * (1 + true_lift))   # treatment adds to cart
        z, p = proportions_ztest([cart_t, cart_c], [n_t, n_c])
        if p < alpha:                                 # test says "significant"
            significant += 1

    return significant / n_runs                       # share of significant tests

power_sim = run_many(true_lift=0.10)
false_alarm_sim = run_many(true_lift=0.0)

print(f"Real +10% effect → test says significant: {power_sim:.1%}  (expected ≈ 80%)")
print(f"No effect        → test says significant: {false_alarm_sim:.1%}  (expected ≈ 5%)")

Real +10% effect → test says significant: 78.5%  (expected ≈ 80%)
No effect        → test says significant: 4.1%  (expected ≈ 5%)


**Findings:**
- **Real +10% effect:** the test said "significant" in **78.5%** of 1,000 runs, close to the planned **80% power**.
- **No effect:** the test said "significant" in **4.1%** of runs, close to the planned **5% false-alarm rate**.

Both are within normal simulation noise, so **the test design works as planned**.
It also explains Step 5: about 1 in 5 tests misses a real +10% effect,
and the seed-42 test was one of those misses.

## Summary
**Question:** Would showing ratings and reviews near "Add to cart" increase add-to-cart?

**Test design:** Baseline 26.64% view → cart (real GA4 data, 45,107 product viewers).
+10% MDE · 5% significance · 80% power → **4,454 users per group (~2 weeks)**.

**One simulated test:** +1.53 pts · p = 0.110 · 95% CI −0.35 to +3.40 pts · guardrail OK
→ **DO NOT SHIP (inconclusive)**.

**1,000 simulated tests:** real +10% effect detected **78.5%** of the time;
false alarms with no effect **4.1%** → the design works as planned.

**What I learned:** "not significant" means "no clear evidence", not "no effect";
the sample size must be decided before the test; and any single test can miss a real effect.

**Limitations:** simulated data; the +10% effect is an assumption; purchase is simulated
as unchanged and independent of add-to-cart; the duration estimate uses holiday-period traffic.